# TensorRT-LLM: NVIDIA's Production Inference Optimizer

> ⚠️ **Untested — Hardware Unavailable**
> 
> TensorRT-LLM requires an **NVIDIA Ampere (A100) or Hopper (H100) GPU** plus the official NVIDIA Docker image.
> It cannot install or run on a T4 (Colab free tier).
> 
> **What this notebook does instead:**
> - Explains TRT-LLM architecture and concepts accurately
> - Runs  + INT8 quantization on GPT-2 as a T4-compatible proxy
> - Provides the full TRT-LLM workflow as reference code (correct syntax, not tested)
> 
> **To run the real TRT-LLM sections:** Use Colab Pro (A100 runtime) or an A100/H100 VM with Docker.

## TensorRT-LLM Architecture

**TensorRT-LLM** (NVIDIA, 2023) is a compile-time optimizer for LLM inference.
Unlike vLLM (which optimizes at runtime), TRT-LLM builds a hardware-specific binary engine once,
then runs it extremely fast.



### What TRT-LLM Does (compile-time)

1. **Graph optimization**: Fuse sequences of ops (bias+gelu, norm+residual) into single kernels
2. **Hardware-specific kernels**: Generate PTX optimized for the exact GPU
3. **Tensor Core scheduling**: Layout matrix multiplications for WMMA instructions
4. **Quantization baking**: INT8/INT4/FP8 logic compiled in — no runtime overhead

### Why A100/H100 Required

- FP8 data type: only on Hopper (H100)
- BF16 Tensor Cores: Ampere+
- NVLink for tensor-parallel: V100 has it but bandwidth too low for 70B+
- The build process validates against `compute_capability >= 8.0` for most paths


In [ ]:
# ── Demonstrate TRT-LLM concepts via torch.compile + INT8 quantization on GPT-2 ──
# torch.compile uses Triton (same fusion technology as TRT-LLM) under the hood

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Compute  : {cc[0]}.{cc[1]}")
    if cc[0] >= 8:
        print("Supports BF16 Tensor Cores (Ampere+)")
    else:
        print("T4/V100: FP16 only (no BF16 Tensor Cores)")

# Load GPT-2 (ungated, 124M params — runnable on any GPU)
print("\nLoading GPT-2 ...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model_fp16 = AutoModelForCausalLM.from_pretrained(
    "gpt2", torch_dtype=torch.float16
).eval().cuda()

n = sum(p.numel() for p in model_fp16.parameters())
m = torch.cuda.memory_allocated() / 1e6
print(f"GPT-2: {n/1e6:.0f}M params, {m:.0f} MB GPU")

In [ ]:
# ── Show engine building concept: INT8 dynamic quantization ──
# This mirrors what TRT-LLM's weight-only INT8 pass does:
# convert Linear layers to quantized equivalents

import copy

def apply_dynamic_int8(model):
    """Quantize all Linear layers to INT8 (simulates TRT-LLM weight-only INT8)."""
    return torch.quantization.quantize_dynamic(
        model.cpu(),
        {nn.Linear},
        dtype=torch.qint8
    )

print("Applying INT8 dynamic quantization (simulating TRT-LLM INT8 weight pass) ...")
model_int8 = apply_dynamic_int8(copy.deepcopy(model_fp16))

# Compare sizes
def model_size_mb(m):
    return sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6

fp16_mb = sum(p.numel() * 2 for p in model_fp16.parameters()) / 1e6
int8_mb = sum(
    p.numel() * (1 if p.dtype == torch.int8 else p.element_size())
    for p in model_int8.parameters()
) / 1e6

print(f"\nFP16 model: ~{fp16_mb:.0f} MB")
print(f"INT8 model: ~{int8_mb:.0f} MB")
print(f"Reduction : {fp16_mb/int8_mb:.1f}x")
print("\nNote: TRT-LLM's INT8 has better kernels (Tensor Core INT8),")
print("but the memory reduction is the same ~2x.")

In [ ]:
# ── Benchmark torch.compile vs eager mode ──
# torch.compile uses the Inductor backend (Triton kernels) — same technology
# underlying TRT-LLM's graph fusion step.

def benchmark_generate(model, tokenizer, prompt, n_tokens=50, warmup=2, runs=5):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}

    for _ in range(warmup):
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=n_tokens,
                               do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(runs):
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=n_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / runs
    toks_per_s = n_tokens / elapsed
    return elapsed, toks_per_s


prompt = "Once upon a time in a land of transformers and attention heads"

# Eager baseline
print("Benchmarking eager mode ...")
t_eager, tp_eager = benchmark_generate(model_fp16, tokenizer, prompt)
print(f"  Eager : {t_eager*1000:.0f} ms / 50 tokens  ({tp_eager:.1f} tok/s)")

# torch.compile
print("Compiling model with torch.compile (first call triggers compilation) ...")
model_compiled = torch.compile(model_fp16, mode="reduce-overhead")
t_compile, tp_compile = benchmark_generate(model_compiled, tokenizer, prompt)
print(f"  Compiled: {t_compile*1000:.0f} ms / 50 tokens  ({tp_compile:.1f} tok/s)")
print(f"  Speedup : {tp_compile/tp_eager:.2f}x")

print("\nNote: torch.compile speedup is typically 1.2–2x on simple GPT-2.")
print("TRT-LLM achieves larger gains (3-5x) because it also uses custom")
print("WMMA kernels and the Transformer Engine on Hopper.")

## What TRT-LLM Does (Reference Code — Not Runnable Here)

The following code shows the actual TRT-LLM workflow. It requires the
`tensorrt_llm` package (only in the official Docker image):

```python
# === STEP 1: Convert HuggingFace checkpoint ===
# (run in shell, not Python)
# python examples/llama/convert_checkpoint.py \
#     --model_dir meta-llama/Llama-2-7b-hf \
#     --output_dir ./llama7b-trt \
#     --dtype float16

# === STEP 2: Build the engine ===
# trtllm-build \
#     --checkpoint_dir ./llama7b-trt \
#     --output_dir ./llama7b-engine \
#     --gemm_plugin float16 \
#     --max_batch_size 64 \
#     --max_input_len 2048 \
#     --max_output_len 512
# Build time: 5-30 min (one-time cost). Engine is GPU-specific.

# === STEP 3: Run inference ===
from tensorrt_llm import LLM as TRTLLM

llm = TRTLLM(
    model_dir     = "./llama7b-engine",
    tokenizer_dir = "meta-llama/Llama-2-7b-hf"
)
outputs = llm.generate(["Hello, world!"], max_new_tokens=100)

# === INT4-AWQ quantization ===
# trtllm-build \
#     --checkpoint_dir ./llama7b-trt \
#     --output_dir ./llama7b-engine-int4 \
#     --use_weight_only \
#     --weight_only_precision int4_awq \
#     --per_group
# Result: 3.5 GB instead of 14 GB for 7B; 3-4x faster decode
```

### Quantization Options (TRT-LLM)

| Method | Memory | Speedup vs FP16 | Notes |
|--------|--------|----------------|-------|
| FP16 | 1x | 1x | Baseline |
| INT8 weight-only | 0.5x | 1.5–2x | No calibration needed |
| INT4-AWQ | 0.25x | 3–4x | Best quality at INT4 |
| FP8 (H100 only) | 0.5x | 2x | Near-FP16 quality |

### TRT-LLM vs vLLM

| Aspect | TRT-LLM | vLLM |
|--------|---------|------|
| Setup | Complex (Docker + build) | `pip install vllm` |
| Latency | Best (~2x better than vLLM) | Good |
| INT4/INT8 | Excellent (WMMA kernels) | Limited |
| Hardware | NVIDIA Ampere+ | Any CUDA GPU |
| Iteration | Slow (rebuild on change) | Fast |
| Best for | Stable production on H100 | Most production use cases |

## TRT-LLM Docker Setup

```bash
# Pull official image (requires NVIDIA NGC account)
docker pull nvcr.io/nvidia/tritonserver:24.01-trtllm-python-py3

# Or use the TRT-LLM release image
docker pull nvcr.io/nvidia/tensorrt_llm/release:0.8.0

# Run with GPU access
docker run --gpus all --rm -it \
    -v $(pwd):/workspace \
    nvcr.io/nvidia/tensorrt_llm/release:0.8.0 \
    bash

# Inside container:
# pip install tensorrt_llm
# python examples/run.py ...
```

Official docs: https://github.com/NVIDIA/TensorRT-LLM

## ✅ Summary

- Explained TRT-LLM architecture: compile-time fusion vs vLLM runtime optimization
- Demonstrated the underlying concepts with `torch.compile` on GPT-2 (actually ran on T4)
- Applied INT8 dynamic quantization showing 2x memory reduction
- Benchmarked eager vs compiled mode
- Showed full TRT-LLM workflow (convert → build → run) as reference code
- Provided Docker setup for A100/H100 environments